<a href="https://colab.research.google.com/github/eduardozurek/ec1/blob/main/_006_nois2_audio/_001_generar_audio_inc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generar audio.inc para Nios II Assembly

Genera solamente `audio.inc`, con una directiva `.word 0xXXXXXXXX` por muestra. No modifica ni concatena datos a una cabecera assembly.

Conserva el procesamiento del notebook original: mono, remuestreo a 8 kHz, recorte de silencio de los extremos con `top_db=40` y primeras 80.000 muestras. Para toda la grabacion cambiar `MAX_MUESTRAS` a `None`. El recorte no elimina pausas internas.

FFmpeg debe estar instalado para decodificar M4A; esta disponible habitualmente en Colab. Se saturan los valores a [-1, 1] antes de convertir a enteros para evitar desbordamientos.


In [ ]:
%pip -q install librosa soundfile audioread


In [ ]:
from pathlib import Path
import librosa
import numpy as np

# Configuracion
ARCHIVO_ENTRADA = Path("e_zurek_contando_hasta_100.m4a")
ARCHIVO_SALIDA = Path("audio.inc")
FRECUENCIA = 8000
MAX_MUESTRAS = 80000  # Igual al notebook original. None para todo el audio.

# En Colab, solicitar el audio si todavia no esta en la carpeta actual.
if not ARCHIVO_ENTRADA.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"Coloque {ARCHIVO_ENTRADA} en la carpeta actual.")
    archivos = files.upload()
    if not ARCHIVO_ENTRADA.exists():
        raise FileNotFoundError(f"Debe subir {ARCHIVO_ENTRADA.name} o ajustar ARCHIVO_ENTRADA.")

# Mismo procesamiento del notebook original.
# Decodificar M4A con FFmpeg; no cambiar aun la frecuencia original.
import subprocess
import tempfile
with tempfile.TemporaryDirectory() as carpeta_temporal:
    wav_temporal = Path(carpeta_temporal) / "decodificado.wav"
    subprocess.run([
        "ffmpeg", "-v", "error", "-y", "-i", str(ARCHIVO_ENTRADA),
        "-c:a", "pcm_f32le", str(wav_temporal)
    ], check=True)
    y, sr = librosa.load(str(wav_temporal), sr=None, mono=True)
y_resampled = librosa.resample(y, orig_sr=sr, target_sr=FRECUENCIA)
y_trimmed, indices = librosa.effects.trim(y_resampled, top_db=40)

# Saturar evita desbordamientos si el remuestreo genera picos fuera de [-1, 1].
y_int32 = (np.clip(y_trimmed.astype(np.float64), -1.0, 1.0)
           * (2**31 - 1)).astype(np.int32)
if MAX_MUESTRAS is not None:
    y_int32 = y_int32[:MAX_MUESTRAS]
if y_int32.size == 0:
    raise ValueError("No hay muestras para exportar.")

# Archivo para .include: una directiva .word por muestra de 32 bits.
# No escribir etiquetas, .data, .end ni una cabecera de assembly.
# El modo w reemplaza la salida anterior y evita acumular muestras.
with ARCHIVO_SALIDA.open("w", encoding="ascii", newline="\n") as f:
    for muestra in y_int32:
        f.write(f"    .word 0x{int(muestra) & 0xFFFFFFFF:08x}\n")

cantidad = int(y_int32.size)
print(f"Archivo: {ARCHIVO_SALIDA}")
print(f"Frecuencia: {FRECUENCIA} Hz")
print(f"Muestras: {cantidad}")
print(f"Duracion: {cantidad / FRECUENCIA:.3f} s")
print(f"Bytes a cargar en RAM: {cantidad * 4} = 0x{cantidad * 4:08x}")
print('Incluir en assembly con: .include "audio.inc"')


## Incorporar audio.inc al programa principal

Guardar `audio.inc` en una carpeta que el ensamblador pueda encontrar. En el programa principal:

```asm
    .data
    .align 2
SONIDO:
    .include "audio.inc"
SONIDO_FIN:
    .end
```

El bucle puede cargar las direcciones con `movia r7, SONIDO` y `movia r8, SONIDO_FIN`, leer cada muestra con `ldw r4, 0(r5)` y avanzar cuatro bytes por muestra.

Agregar solo el programa principal como unidad de compilacion: `audio.inc` se incorpora con `.include`. Si el ensamblador no lo encuentra, agregar su carpeta a las rutas del ensamblador mediante `-I`.

El archivo se abre en modo `w`: al volver a ejecutar, reemplaza el contenido anterior sin acumular muestras. No contiene `.end`, porque esa directiva debe quedar al final del programa principal.

Las 80.000 muestras ocupan 320.000 bytes (`0x0004E200`) de datos. Este metodo incluye el audio en el ejecutable; no se carga por separado mediante Load file into memory. Configurar la reproduccion del dispositivo de audio a 8 kHz.


Puedes crear una celda con el siguiente código para descargar los datos desde colab:

```python
# Descarga opcional en Google Colab.
try:
    from google.colab import files
except ImportError:
    print(f"Archivo disponible en: {ARCHIVO_SALIDA.resolve()}")
else:
    files.download(str(ARCHIVO_SALIDA))
```